# Sujet : Impact des espaces verts publics sur le prix au mètre carré de l'immobilier à Paris

*Projet Python pour la Data Science - mettre les noms*

## Introduction

rédiger l'intro

### Sommaire

- **Introduction**

- **I. Création et exploration de la base de données principale**
    - A. Récupération des données
        - _0. Packages et paramètres_
        - _1. Espaces verts parisiens_
        - _2. Ventes immobilières à Paris_
        - _3. Autres bases explorées 
    - B. Nettoyage des données
        - _1. Sélection des données_
        - _2. Harmonisation des types_
        - _3. Valeurs manquantes et atypiques_
        - _4. Variables utiles pour la régression_
        - _5. Harmonisation des noms de colonnes_
        - _6. Création de couches géométriques_
    - C. Export de la base finale

- **II. Visualisations et statistiques descriptives**
    - A. Exploration de la base
        - _1. ..._
        - _2. ..._
    - B. Visualisation des données
    - C. ...

- **III. Modélisation : effet des espaces verts sur le prix du mètre carré à Paris**
    - A. Objectif
    - B. 
    - C
    - D. Conclusion de la modélisation

- **Conclusion**

# I. Création et exploration de la base de données principale

Dans cette partie, nous importons les données brutes, les nettoyons, puis nous les croisons spatialement pour obtenir une base exploitable : **une ligne par vente d'appartement à Paris**, enrichie d'indicateurs d'accès aux espaces verts.

| Fichier brut | Description | Source | Mode d'extraction |
|---|---|---|---|
| `espaces_verts.csv` | Inventaire des espaces verts et assimilés de la Ville de Paris : nom, typologie, catégorie, surfaces, contour géographique. | Ville de Paris, Direction des Espaces Verts et de l'Environnement ([lien](https://www.data.gouv.fr/datasets/espaces-verts-et-assimiles/)). | CSV (séparateur `;`) |
| `espaces_verts_supplementaires.csv` | Surfaces d'espaces verts créées par projet. | Ville de Paris ([lien](https://www.data.gouv.fr/datasets/m2-despaces-verts-supplementaires )). | CSV (séparateur `;`) |
| `PJ_75_2021.gpkg` | Parcs et jardins ouverts au public à Paris (millésime 2021), sous forme de polygones. | **[source à préciser]** | GeoPackage |
| `dvf_plus_d75.csv` | Mutations immobilières du département de Paris (DVF+). | Cerema, DVF+, construit à partir des Demandes de valeurs foncières de la DGFiP ([lien](https://doc-datafoncier.cerema.fr/)). | CSV (séparateur `\|`), téléchargé depuis un Drive partagé avec `gdown` |

## A. Récupération des données

### 0. Imports et paramètres

In [51]:
import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import gdown
from shapely.geometry import shape

pd.set_option("display.max_columns", None)   # afficher toutes les colonnes

# Se placer à la racine du projet (le dossier qui contient "data"), quel que soit le poste
if not os.path.exists("data") and os.path.exists("../data"):
    os.chdir("..")
print("Dossier de travail :", os.getcwd())

# Paramètres
FILE_DVF_PLUS_ID = "1Mnsnm-OCvw-5zLuVBf6p41NtDChE_yg1"   # identifiant du fichier sur le Drive partagé
FICHIER_LOCAL_DVF_PLUS = "dvf_plus_d75.csv"               # fichier volumineux : à mettre dans le .gitignore
DOSSIER_BRUT = "data/1.data_brute"
DOSSIER_SORTIE = "data/2.data_nettoyee"

Dossier de travail : /Users/clothildemigaud/Projet python 2A /Projet-python-2A-


### 1. Espaces verts parisiens

In [52]:
ev_actuels = pd.read_csv(f"{DOSSIER_BRUT}/espaces_verts.csv", sep=";", encoding="utf-8")
ev_supp = pd.read_csv(f"{DOSSIER_BRUT}/espaces_verts_supplementaires.csv", sep=";", encoding="utf-8")

n_ev_brut = len(ev_actuels)
print(f"Espaces verts (brut) : {ev_actuels.shape[0]} lignes, {ev_actuels.shape[1]} colonnes")
print(f"Espaces verts supplémentaires (brut) : {ev_supp.shape[0]} lignes, {ev_supp.shape[1]} colonnes")
display(ev_actuels.head())
display(ev_supp.head())

Espaces verts (brut) : 2535 lignes, 31 colonnes
Espaces verts supplémentaires (brut) : 64 lignes, 6 colonnes


,Identifiant espace vert,Nom de l'espace vert,Typologie d'espace vert,Catégorie,Adresse - Numéro,Adresse - Complément,Adresse - type voie,Adresse - Libellé voie,Code postal,Surface calculée,Superficie totale réelle,Surface horticole,Présence cloture,Périmètre,Année de l'ouverture,Année de rénovation,Ancien nom de l'espace vert,Année de changement de nom,Nombre d'entités,Ouverture 24h_24h,ID_DIVISION,ID_ATELIER_HORTICOLE,IDA3D_ENB,SITE_VILLES,ID_EQPT,Compétence,Geo Shape,URL_PLAN,Geo point,last_edited_user,last_edited_date
0,655.0,JARDIN LUC HOFFMANN,Promenades ouvertes,Jardin,10.0,NaN,RUE DU,MAROC,75019.0,7355.0,6638.0,1897.0,Oui,445.021177,1967.0,1975.0,SQUARE TANGER - MAROC - FLANDRE,NaN,1.0,Non,119.0,42.0,66382,349,3702,CA,"{""coordinates"": [[[2.3702529557340135, 48.8869...",https://b22-pr-v1-iis01.ressources.paris.mdp/M...,"48.88674624509649, 2.3705036049494175",NaN,NaN
1,10338.0,JARDINIERE DU PALAIS DE LA PORTE DOREE,Végétalisation d'espace public,Jardiniere,2.0,NaN,AVENUE,ARMAND ROUSSEAU,75012.0,65.0,80.0,80.0,Non,48.631662,NaN,NaN,JARDINIERE DU MUSEE DES ARTS AFRICAINS ET OCEA...,NaN,1.0,NaN,111.0,16.0,JDE7285,SV,7285,CA,"{""coordinates"": [[[2.4084647306259677, 48.8352...",https://b22-pr-v1-iis01.ressources.paris.mdp/M...,"48.83510364192792, 2.408483380584031",NaN,NaN
2,319.0,03-08,Périphérique,Talus,0.0,NaN,NaN,NaN,75014.0,924.0,NaN,NaN,Non,175.780126,NaN,NaN,NaN,NaN,1.0,NaN,9.0,NaN,0,0,0,0,"{""coordinates"": [[[2.3435270948961424, 48.8161...",https://b22-pr-v1-iis01.ressources.paris.mdp/M...,"48.816309504786766, 2.3433235945845063",NaN,NaN
3,769.0,JARDIN DU 60 RUE DE LA JUSTICE,Promenades ouvertes,Jardin,60.0,NaN,RUE DE LA,JUSTICE,75020.0,431.0,416.0,80.0,Oui,101.207419,1980.0,NaN,NaN,NaN,1.0,Non,120.0,45.0,68846,3419,3381,CA,"{""coordinates"": [[[2.4083937552062666, 48.8681...",https://b22-pr-v1-iis01.ressources.paris.mdp/M...,"48.86829230387095, 2.40853055999283",NaN,NaN
4,10706.0,JARDINIERES DE LA RUE DE BUZENVAL,Végétalisation d'espace public,Jardiniere,105.0,NaN,RUE DE,BUZENVAL,75020.0,179.0,185.0,185.0,Non,120.120629,2006.0,NaN,NaN,NaN,2.0,NaN,120.0,46.0,JDE7589,SV,7589,CA,"{""coordinates"": [[[[2.398858670258213, 48.8548...",https://b22-pr-v1-iis01.ressources.paris.mdp/M...,"48.854952829312936, 2.3988870952147727",NaN,NaN


,Nom du projet,Arrondissement,Secteur administratif,Date de fin de travaux,Type d'opération,M2 d’espaces verts supplémentaires
0,Transformation de la cour de l’espace Gabriel ...,12,12,2025,Nouvelle surface d’espaces verts,830.0
1,Végétalisation du Centre Sportif Elisabeth,14,14,2023,Nouvelle surface d’espaces verts,243.0
2,Renaturation du square Antoine Blondin,20,20,2022,Nouvelle surface d’espaces verts,949.0
3,Renaturation et extension du square des Périch...,15,15,2021,Nouvelle surface d’espaces verts,860.0
4,Ouverture au public PC18 - Bvd Ornano / rue de...,18,18,2024,Nouvelle surface d’espaces verts,13440.0


In [53]:
ev_actuels.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2535 entries, 0 to 2534
Data columns (total 31 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Identifiant espace vert      2526 non-null   float64
 1   Nom de l'espace vert         2535 non-null   object 
 2   Typologie d'espace vert      2535 non-null   object 
 3   Catégorie                    2527 non-null   object 
 4   Adresse - Numéro             2529 non-null   float64
 5   Adresse - Complément         373 non-null    object 
 6   Adresse - type voie          2287 non-null   object 
 7   Adresse - Libellé voie       2280 non-null   object 
 8   Code postal                  2532 non-null   float64
 9   Surface calculée             1943 non-null   float64
 10  Superficie totale réelle     1955 non-null   float64
 11  Surface horticole            1877 non-null   float64
 12  Présence cloture             2245 non-null   object 
 13  Périmètre         

Les espaces verts sont décrits par une **typologie** et une **catégorie**. Nous les explorons pour décider lesquels conserver.

In [54]:
display(ev_actuels["Typologie d'espace vert"].value_counts(dropna=False))
display(
    ev_actuels.groupby(["Typologie d'espace vert", "Catégorie"], dropna=False)
    .size().rename("nb").reset_index()
    .sort_values(["Typologie d'espace vert", "nb"], ascending=[True, False])
)

Typologie d'espace vert
Végétalisation d'espace public       1013
Promenades ouvertes                   598
Murs végétalisés                      347
Périphérique                          263
Ephémères, partagés, pédagogiques     142
Jardinets décoratifs                  117
Cimetières                             21
Etablissements sportifs                19
Jardins privatifs                      13
Bois                                    2
Name: count, dtype: int64

,Typologie d'espace vert,Catégorie,nb
0,Bois,Bois,2
1,Cimetières,Cimetière,21
2,"Ephémères, partagés, pédagogiques",Jardin partage,142
3,Etablissements sportifs,Espace Vert,19
6,Jardinets décoratifs,Jardinet,75
10,Jardinets décoratifs,Plate-bande,10
12,Jardinets décoratifs,Square,10
13,Jardinets décoratifs,Talus,7
9,Jardinets décoratifs,Pelouse,5
5,Jardinets décoratifs,Jardin,4


### 2. Ventes immobilières à Paris

Le fichier DVF+ est volumineux : il est téléchargé une seule fois depuis le Drive partagé, puis lu en local.

In [55]:
if not os.path.exists(FICHIER_LOCAL_DVF_PLUS):
    gdown.download(id=FILE_DVF_PLUS_ID, output=FICHIER_LOCAL_DVF_PLUS, quiet=False)
dvf_plus = pd.read_csv(FICHIER_LOCAL_DVF_PLUS, sep="|", low_memory=False)

n_dvf_brut = len(dvf_plus)
print(f"DVF+ (brut) : {dvf_plus.shape[0]} lignes, {dvf_plus.shape[1]} colonnes")

DVF+ (brut) : 519013 lignes, 63 colonnes


In [56]:
display(dvf_plus["anneemut"].value_counts().sort_index())
display(dvf_plus["libnatmut"].value_counts(dropna=False))
display(dvf_plus["codtypbien"].value_counts(dropna=False).head(10))

anneemut
2014    35612
2015    41183
2016    39977
2017    45843
2018    44682
2019    46064
2020    38155
2021    43564
2022    47064
2023    39297
2024    36028
2025    41435
2026    20109
Name: count, dtype: int64

libnatmut
Vente                                 507876
Vente en l'état futur d'achèvement      7429
Echange                                 2273
Adjudication                            1377
Vente terrain à bâtir                     43
Expropriation                             15
Name: count, dtype: int64

codtypbien
121    375041
131     59743
14      33608
122     18243
152      8730
132      8425
120      7126
101      4292
111      1822
102      1397
Name: count, dtype: int64

### 3. Autres bases explorées

Nous avons aussi exploré le fichier des **parcs et jardins ouverts au public**, qui permet de distinguer les espaces réellement accessibles au public et de regrouper les parcs découpés en plusieurs polygones.

In [ ]:
parcs_publics = gpd.read_file(f"{DOSSIER_BRUT}/PJ_75_2021.gpkg")
print("Système de coordonnées :", parcs_publics.crs)
parcs_publics.info()
display(parcs_publics.head())

## B. Nettoyage des données

### 1. Sélection des données

**Espaces verts.** Nous ne gardons que les espaces situés à Paris (codes postaux commençant par 75) et les typologies et catégories qui correspondent à de vrais espaces de verdure.

In [ ]:
# Paris uniquement
ev_actuels = ev_actuels[ev_actuels["Code postal"].astype("string").str.startswith("75", na=False)].copy()

categories_to_keep = [
    "Jardin", "Square", "Promenade", "Espace Vert", "Jardin partage", "Parc", "Pelouse",
    "Arboretum", "Mail", "Jardin d'immeubles", "Archipel", "Foret urbaine",
    "Terrain de boules", "jardiniere", "Jardiniere", "Cimetière", "Bois", "Ile",
]   # 'Murs vegetalises' est volontairement exclu
types_to_keep = [
    "Bois", "Ephémères, partagés, pédagogiques", "Jardinets décoratifs",
    "Promenades ouvertes", "Végétalisation d'espace public",
]

ev_actuels = ev_actuels[ev_actuels["Typologie d'espace vert"].isin(types_to_keep)]
ev_actuels = ev_actuels[ev_actuels["Catégorie"].isin(categories_to_keep)].copy()

print(f"Espaces verts conservés : {len(ev_actuels)} sur {n_ev_brut}")

**Parcs et jardins ouverts au public.** Un même parc peut être découpé en plusieurs polygones portant le même nom : nous les fusionnons pour obtenir une ligne par parc.

In [ ]:
occurrences = parcs_publics["nom"].value_counts()
noms_a_fusionner = occurrences[occurrences > 1].index

a_fusionner = parcs_publics[parcs_publics["nom"].isin(noms_a_fusionner)]
a_conserver = parcs_publics[~parcs_publics["nom"].isin(noms_a_fusionner)]
fusionnes = a_fusionner.dissolve(by="nom", as_index=False)

parcs_publics_nettoyee = gpd.GeoDataFrame(
    pd.concat([fusionnes, a_conserver], ignore_index=True), crs=parcs_publics.crs
)

# Surface en m² (calculée dans un système de coordonnées en mètres : Lambert 93)
parcs_publics_nettoyee["surface_m2"] = parcs_publics_nettoyee.to_crs("EPSG:2154").geometry.area

print(f"Parcs et jardins : {len(parcs_publics)} polygones -> {len(parcs_publics_nettoyee)} espaces après fusion")

**Ventes immobilières.** Nous ne gardons que les ventes d'**un seul appartement**, sur une seule parcelle :
- les maisons sont très peu nombreuses à Paris et leur prix dépend aussi du terrain, ce qui les rend peu comparables aux appartements ;
- le prix de vente (`valeurfonc`) est global : il porte sur tous les lots de la mutation, sans répartition entre eux. Les ventes mêlant plusieurs types de biens, communes, dispositions ou parcelles sont donc écartées ;
- nous conservons pour l'instant les ventes avec dépendances (cave, parking) : deux bases seront produites à la fin.

In [ ]:
dvf_plus = dvf_plus[dvf_plus['anneemut'].between(2023, 2025)] #année de mutation
dvf_plus = dvf_plus[dvf_plus['libnatmut'] == 'Vente'] #on ne garde que les ventes (on exclut "vente en l'état futur d'achèvement" et "échange")
dvf_plus = dvf_plus[dvf_plus['nblocapt'] == 1] # nombre d'appartements ayant muté 
dvf_plus = dvf_plus[dvf_plus['nblocmai'] == 0] # nombre de maisons ayant muté 
dvf_plus = dvf_plus[dvf_plus['nblocact'] == 0] # nombre de locaux d'activité ayant muté
dvf_plus = dvf_plus[dvf_plus['codtypbien'] == 121]  # Un appartement, selon cette livraison

# on évite les ventes complexes (plusieurs communes, dispositions ou parcelles)
dvf_plus = dvf_plus[dvf_plus['nbcomm'] == 1] # nombre de communes 
dvf_plus = dvf_plus[dvf_plus['nbdispo'] == 1] # nombre de dispositions associées à la mutation (98% des mutations ont une seule disposition)
dvf_plus = dvf_plus[dvf_plus['nbpar'] == 1] # nombre de parcelles 

dvf_plus = dvf_plus.copy()

### 2. Harmonisation des types

Nous supprimons les colonnes techniques inutiles, puis nous convertissons les colonnes numériques en entiers (qui acceptent les valeurs manquantes).

In [ ]:
col_to_drop = [
    (ev_actuels, ["Identifiant espace vert", "Présence cloture", "Année de rénovation",
                  "Ancien nom de l'espace vert", "Année de changement de nom", "Nombre d'entités",
                  "Ouverture 24h_24h", "ID_DIVISION", "ID_ATELIER_HORTICOLE", "IDA3D_ENB",
                  "ID_EQPT", "Compétence", "last_edited_user", "last_edited_date"]),
    (ev_supp, ["Type d'opération", "Secteur administratif"]),
]
for df, cols in col_to_drop:
    df.drop(columns=cols, inplace=True)

col_to_integer = [
    (ev_actuels, ["Adresse - Numéro", "Code postal", "Surface calculée", "Superficie totale réelle",
                  "Surface horticole", "Année de l'ouverture"]),
    (ev_supp, ["M2 d’espaces verts supplémentaires"]),
]
for df, cols in col_to_integer:
    for col in cols:
        df[col] = df[col].astype("Int64")

ev_actuels.info()

### 3. Valeurs manquantes et atypiques

**Espaces verts.** Certaines valeurs sont en réalité des codes d'absence de donnée : `0` pour un numéro d'adresse, `9999` pour une année d'ouverture. Nous les remplaçons par des valeurs manquantes.

In [ ]:
ev_actuels["Adresse - Numéro"] = ev_actuels["Adresse - Numéro"].replace(0, np.nan)
ev_actuels["Année de l'ouverture"] = ev_actuels["Année de l'ouverture"].replace(9999, np.nan)

display(ev_actuels.describe())

**Ventes.** Nous retirons les ventes sans prix, sans surface ou sans coordonnées, puis les valeurs atypiques :
- surface inférieure ou égale à 9 m² ;
- prix au m² en dehors de la fourchette 1 000 à 30 000 €.

Ces seuils sont choisits à partir de la distribution affichée ci dessous 

In [58]:

dvf_plus["prix_m2"] = dvf_plus["valeurfonc"] / dvf_plus["sbatapt"]
display(dvf_plus["prix_m2"].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]))

necessaires = ['valeurfonc', 'sbatapt', 'geompar_x', 'geompar_y']
dvf_plus = dvf_plus.dropna(subset=necessaires).copy()
dvf_plus = dvf_plus[dvf_plus['valeurfonc'] > 0].copy()
dvf_plus = dvf_plus[dvf_plus['sbatapt'] > 9].copy() # chat voulait limiter à 300 m2 ??

dvf_plus['prix_m2'] = dvf_plus['valeurfonc'] / dvf_plus['sbatapt'] # on rajoute une colonne px_m2
dvf_plus = dvf_plus[dvf_plus['prix_m2'].between(1000, 30000)].copy()                                      


count     81827.000000
mean      10046.080971
std        4061.152649
min           0.004348
1%           66.666667
5%         4700.000000
25%        8205.128205
50%        9772.388060
75%       11589.538462
95%       15862.068966
99%       22826.086957
max      193846.153846
Name: prix_m2, dtype: float64

### 4. Variables utiles pour la régression

In [59]:
dvf_plus["avec_dependance"] = (dvf_plus["nblocdep"] > 0).astype(int)

# nombre de pièces de l'appartement (la modalité 5 signifie 5 pièces ou plus)
dvf_plus["nb_pieces"] = (
    dvf_plus["nbapt1pp"] + 2 * dvf_plus["nbapt2pp"] + 3 * dvf_plus["nbapt3pp"]
    + 4 * dvf_plus["nbapt4pp"] + 5 * dvf_plus["nbapt5pp"]
)

dvf_plus["log_prix_m2"] = np.log(dvf_plus["prix_m2"])
dvf_plus["log_surface"] = np.log(dvf_plus["sbatapt"])

dvf_plus[["prix_m2", "sbatapt", "nb_pieces", "avec_dependance", "log_prix_m2", "log_surface"]].describe()

,prix_m2,sbatapt,nb_pieces,avec_dependance,log_prix_m2,log_surface
count,79367.000000,79367.000000,79367.000000,79367.000000,79367.000000,79367.000000
mean,10219.571632,52.152746,2.334509,0.661383,9.181682,3.751437
std,3210.846871,37.324626,1.157397,0.473242,0.332820,0.631041
min,1000.000000,10.000000,1.000000,0.000000,6.907755,2.302585
25%,8333.333333,28.000000,1.000000,0.000000,9.028019,3.332205
50%,9848.484848,42.000000,2.000000,1.000000,9.195073,3.737670
75%,11637.931034,66.000000,3.000000,1.000000,9.362025,4.189655
max,30000.000000,1500.000000,5.000000,1.000000,10.308953,7.313220


### 5. Harmonisation des noms de colonnes

Nous ne gardons que les variables utiles et leur donnons des noms simples.

In [60]:
colonnes = {
    "idmutation": "id_vente",
    "datemut": "date",
    "anneemut": "annee",
    "moismut": "mois",
    "l_codinsee": "code_arrondissement",
    "l_idpar": "id_parcelle",
    "valeurfonc": "prix_euros",
    "sbatapt": "surface_m2",
    "prix_m2": "prix_m2",
    "nblocdep": "nb_dependances",
    "avec_dependance": "avec_dependance",
    "nb_pieces": "nb_pieces",
    "geompar_x": "x_lambert93",
    "geompar_y": "y_lambert93",
    "log_prix_m2": "log_prix_m2",
    "log_surface": "log_surface",
}
manquantes = [c for c in colonnes if c not in dvf_plus.columns]
assert not manquantes, f"Colonnes absentes : {manquantes}"

dvf_plus = dvf_plus[list(colonnes)].rename(columns=colonnes)
dvf_plus = dvf_plus.sort_values(["date", "id_vente"]).reset_index(drop=True)

# vérifications
assert dvf_plus["id_vente"].is_unique, "Attention : ventes en double"
assert dvf_plus["prix_m2"].gt(0).all(), "Attention : prix au m² invalide"

print(dvf_plus.shape)
dvf_plus.head()

(79367, 16)


,id_vente,date,annee,mois,code_arrondissement,id_parcelle,prix_euros,surface_m2,prix_m2,nb_dependances,avec_dependance,nb_pieces,x_lambert93,y_lambert93,log_prix_m2,log_surface
0,11934819,2023-01-02,2023,1,75108,75108000CG0019,468000.0,60.0,7800.000000,1,1,2,649684.423970,6.864692e+06,8.961879,4.094345
1,12123558,2023-01-02,2023,1,75117,75117000CN0066,895000.0,89.0,10056.179775,1,1,3,650528.612950,6.865361e+06,9.215943,4.488636
2,12199545,2023-01-02,2023,1,75116,75116000DS0036,1245161.4,70.0,17788.020000,1,1,3,647589.231097,6.862543e+06,9.786280,4.248495
3,13155498,2023-01-02,2023,1,75115,75115000CS0085,276000.0,27.0,10222.222222,0,0,2,649736.743154,6.860611e+06,9.232319,3.295837
4,11840368,2023-01-03,2023,1,75111,75111000AB0060,370000.0,27.0,13703.703704,1,1,2,653889.863732,6.863231e+06,9.525421,3.295837


### 6. Création de couches géométriques

Pour croiser les deux sources, nous les transformons en **couches géographiques**, toutes deux en Lambert 93 (système en mètres, adapté aux calculs de distance) :
- les **espaces verts** deviennent des polygones. Leurs contours sont fournis en latitude/longitude (WGS84, `EPSG:4326`) : ils sont **convertis** en Lambert 93 (`EPSG:2154`) ;
- les **ventes** deviennent des points. Leurs coordonnées sont déjà en Lambert 93 : il n'y a pas de conversion, on déclare seulement le système.

In [61]:
# identifiant unique des espaces verts (conservé dans le CSV exporté pour pouvoir faire le lien)
ev_actuels = ev_actuels.reset_index(drop=True)
ev_actuels["id_ev"] = ev_actuels.index

# --- espaces verts : polygones ---
ev = ev_actuels.dropna(subset=["Geo Shape"]).copy()
ev["geometry"] = ev["Geo Shape"].apply(
    lambda s: shape(json.loads(s) if isinstance(s, str) else s)
)
gdf_ev = gpd.GeoDataFrame(ev, geometry="geometry", crs="EPSG:4326").to_crs("EPSG:2154")

# --- ventes : points (créés après le reset_index, donc même index que dvf_plus) ---
gdf_dvf = gpd.GeoDataFrame(
    dvf_plus,
    geometry=gpd.points_from_xy(dvf_plus["x_lambert93"], dvf_plus["y_lambert93"]),
    crs="EPSG:2154",
)

print(f"{len(gdf_ev)} espaces verts (sur {len(ev_actuels)}) et {len(gdf_dvf)} ventes")

2534 espaces verts (sur 2535) et 79367 ventes


#### Espace vert le plus proche de chaque logement

- Nous écartons les très petits espaces verts (seuil provisoire, **[à justifier]**).
- La distance est mesurée entre le logement et le **point le plus proche** du contour de l'espace vert (0 si le logement est dans l'espace vert). Elle est à vol d'oiseau : elle sous-estime le trajet réel à pied.

In [62]:
SEUIL_SURFACE_M2 = 80
gdf_ev = gdf_ev[gdf_ev["Surface calculée"].fillna(0) >= SEUIL_SURFACE_M2].copy()
print(f"{len(gdf_ev)} espaces verts d'au moins {SEUIL_SURFACE_M2} m²")

ev_proche = gpd.sjoin_nearest(
    gdf_dvf[["geometry"]],
    gdf_ev[["id_ev", "Typologie d'espace vert", "Nom de l'espace vert", "Catégorie", "Surface calculée", "geometry"]],
    how="left",
    distance_col="dist_ev_m",
)
# une seule ligne par logement (en cas d'égalité de distance), dans le même ordre que dvf_plus
ev_proche = ev_proche[~ev_proche.index.duplicated(keep="first")].reindex(dvf_plus.index)

dvf_plus["id_ev_proche"] = ev_proche["id_ev"]
dvf_plus["typologie_ev_proche"] = ev_proche["Typologie d'espace vert"]
dvf_plus["categorie_ev_proche"] = ev_proche["Catégorie"]
dvf_plus["nom_ev_proche"] = ev_proche["Nom de l'espace vert"]
dvf_plus["surface_ev_proche"] = ev_proche["Surface calculée"]
dvf_plus["dist_ev_m"] = ev_proche["dist_ev_m"]

dvf_plus["dist_ev_m"].describe()

1515 espaces verts d'au moins 80 m²


count    79367.000000
mean       118.952011
std         83.186059
min          0.000000
25%         56.258819
50%        100.863358
75%        160.469716
max        535.709109
Name: dist_ev_m, dtype: float64

#### Espaces verts dans un rayon autour du logement

Pour chaque logement, nous listons les espaces verts situés dans trois rayons (distances à vol d'oiseau). Chaque colonne contient un dictionnaire `{nom: {distance, surface}}` trié par distance, et une colonne donne leur nombre.

À titre de repère, on compte environ 80 m par minute à pied : **100 m** ≈ 1 min, **200 m** ≈ 2 à 3 min, **300 m** ≈ 4 min, ce dernier seuil étant celui qui est couramment cité comme distance d'accès recommandée (**[à vérifier dans la source avant de le citer]**).

In [63]:
RAYONS = [100, 200, 300]

ev_g = gdf_ev[["id_ev", "Nom de l'espace vert", "Surface calculée", "geometry"]].copy()
ev_g["nom"] = ev_g["Nom de l'espace vert"].fillna("Sans nom")

# 1. tous les couples (logement, espace vert) à moins du plus grand rayon
zone = gdf_dvf[["geometry"]].copy()
zone["geometry"] = zone.geometry.buffer(max(RAYONS))
paires = gpd.sjoin(zone, ev_g, how="inner", predicate="intersects")

# 2. distance exacte de chaque couple (jusqu'au point le plus proche du contour)
paires["dist_m"] = gdf_dvf.geometry.loc[paires.index].distance(
    ev_g.geometry.loc[paires["index_right"]], align=False
)

# 3. une colonne par rayon : dictionnaire {nom: {"dist_m", "surface_m2"}} et nombre d'espaces verts
for r in RAYONS:
    p = paires[paires["dist_m"] <= r]

    # un parc découpé en plusieurs polygones = une seule entrée (distance minimale)
    d = (p.groupby([p.index, "nom"])
           .agg(dist=("dist_m", "min"), surface=("Surface calculée", "max"))
           .reset_index())
    d.columns = ["idx", "nom", "dist", "surface"]
    d = d.sort_values(["idx", "dist"])

    dicts = {}
    for idx, nom, dist, surf in zip(d["idx"], d["nom"], d["dist"], d["surface"]):
        dicts.setdefault(idx, {})[nom] = {
            "dist_m": int(round(dist)),
            "surface_m2": None if pd.isna(surf) else int(round(surf)),
        }

    dvf_plus[f"ev_{r}m"] = [dicts.get(i, {}) for i in dvf_plus.index]
    dvf_plus[f"nb_ev_{r}m"] = dvf_plus[f"ev_{r}m"].apply(len)

dvf_plus[[f"nb_ev_{r}m" for r in RAYONS]].describe()

,nb_ev_100m,nb_ev_200m,nb_ev_300m
count,79367.000000,79367.000000,79367.000000
mean,0.792256,2.575126,5.334912
std,1.006598,2.038030,3.288639
min,0.000000,0.000000,0.000000
25%,0.000000,1.000000,3.000000
50%,0.000000,2.000000,5.000000
75%,1.000000,4.000000,7.000000
max,7.000000,15.000000,23.000000


**Vérification.** Aucun logement ne doit rester sans espace vert le plus proche, et les distances doivent être plausibles.

In [64]:
print("Logements sans espace vert proche :", dvf_plus["dist_ev_m"].isna().sum())
print("Index aligné :", dvf_plus.index.equals(ev_proche.index))
display(dvf_plus[["prix_m2", "dist_ev_m", "nb_ev_100m", "nb_ev_200m", "nb_ev_300m"]].describe())
dvf_plus.head()

Logements sans espace vert proche : 0
Index aligné : True


,prix_m2,dist_ev_m,nb_ev_100m,nb_ev_200m,nb_ev_300m
count,79367.000000,79367.000000,79367.000000,79367.000000,79367.000000
mean,10219.571632,118.952011,0.792256,2.575126,5.334912
std,3210.846871,83.186059,1.006598,2.038030,3.288639
min,1000.000000,0.000000,0.000000,0.000000,0.000000
25%,8333.333333,56.258819,0.000000,1.000000,3.000000
50%,9848.484848,100.863358,0.000000,2.000000,5.000000
75%,11637.931034,160.469716,1.000000,4.000000,7.000000
max,30000.000000,535.709109,7.000000,15.000000,23.000000


,id_vente,date,annee,mois,code_arrondissement,id_parcelle,prix_euros,surface_m2,prix_m2,nb_dependances,avec_dependance,nb_pieces,x_lambert93,y_lambert93,log_prix_m2,log_surface,id_ev_proche,typologie_ev_proche,categorie_ev_proche,nom_ev_proche,surface_ev_proche,dist_ev_m,ev_100m,nb_ev_100m,ev_200m,nb_ev_200m,ev_300m,nb_ev_300m
0,11934819,2023-01-02,2023,1,75108,75108000CG0019,468000.0,60.0,7800.000000,1,1,2,649684.423970,6.864692e+06,8.961879,4.094345,527,Promenades ouvertes,Parc,PARC MONCEAU,82667.0,149.411140,{},0,"{'PARC MONCEAU': {'dist_m': 149, 'surface_m2':...",1,"{'PARC MONCEAU': {'dist_m': 149, 'surface_m2':...",3
1,12123558,2023-01-02,2023,1,75117,75117000CN0066,895000.0,89.0,10056.179775,1,1,3,650528.612950,6.865361e+06,9.215943,4.488636,514,Promenades ouvertes,Square,SQUARE HELENE,875.0,25.837890,"{'SQUARE HELENE': {'dist_m': 26, 'surface_m2':...",2,"{'SQUARE HELENE': {'dist_m': 26, 'surface_m2':...",5,"{'SQUARE HELENE': {'dist_m': 26, 'surface_m2':...",8
2,12199545,2023-01-02,2023,1,75116,75116000DS0036,1245161.4,70.0,17788.020000,1,1,3,647589.231097,6.862543e+06,9.786280,4.248495,581,Promenades ouvertes,Jardin,JARDINS DU TROCADERO,105555.0,45.523257,"{'JARDINS DU TROCADERO': {'dist_m': 46, 'surfa...",2,"{'JARDINS DU TROCADERO': {'dist_m': 46, 'surfa...",7,"{'JARDINS DU TROCADERO': {'dist_m': 46, 'surfa...",8
3,13155498,2023-01-02,2023,1,75115,75115000CS0085,276000.0,27.0,10222.222222,0,0,2,649736.743154,6.860611e+06,9.232319,3.295837,2466,Jardinets décoratifs,Plate-bande,PLATE-BANDE DU BOULEVARD PASTEUR,812.0,195.351652,{},0,{'PLATE-BANDE DU BOULEVARD PASTEUR': {'dist_m'...,2,{'PLATE-BANDE DU BOULEVARD PASTEUR': {'dist_m'...,3
4,11840368,2023-01-03,2023,1,75111,75111000AB0060,370000.0,27.0,13703.703704,1,1,2,653889.863732,6.863231e+06,9.525421,3.295837,2348,Promenades ouvertes,Promenade,PROMENADE RICHARD LENOIR TERRAIN DE BOULES JUL...,836.0,192.958227,{},0,{'PROMENADE RICHARD LENOIR TERRAIN DE BOULES J...,2,{'PROMENADE RICHARD LENOIR TERRAIN DE BOULES J...,5


## C. Export de la base finale

Les colonnes qui contiennent des dictionnaires sont converties en texte JSON pour l'export (elles se relisent avec `json.loads`). Deux versions de la base sont produites : une **complète** et une **sans dépendances** (cave, parking), car le prix de vente ne distingue pas la part de chaque lot.

In [65]:
os.makedirs(DOSSIER_SORTIE, exist_ok=True)

dvf_plus.to_csv(os.path.join(DOSSIER_SORTIE, 'dvf_paris_nettoyee.csv'), index=False)
ev_actuels.to_csv(os.path.join(DOSSIER_SORTIE, 'ev_actuels_nettoyes.csv'), index=False)
ev_supp.to_csv(os.path.join(DOSSIER_SORTIE, 'ev_supp_nettoyes.csv'), index=False)
parcs_publics_nettoyee.to_file("data/2.data_nettoyee/parcs_publics_nettoyee.gpkg", layer="parcs_publics_nettoyee", driver="GPKG")
# version sans dépendances 
sans_dependances = dvf_plus[dvf_plus['nb_dependances'] == 0].copy()
sans_dependances.to_csv(os.path.join("data/2.data_nettoyee", 'dvf_paris_sans_dependances.csv'), index=False)

print("Fichiers enregistrés dans", DOSSIER_SORTIE)
pd.DataFrame({
    "Fichier": ["dvf_paris_nettoyee", "dvf_paris_sans_dependances", "ev_actuels_nettoyes", "ev_supp_nettoyes", "parcs_publics_nettoyee"],
    "Lignes": [len(dvf_plus), len(sans_dependances), len(ev_actuels), len(ev_supp), len(parcs_publics_nettoyee)],
})

Fichiers enregistrés dans data/2.data_nettoyee


,Fichier,Lignes
0,dvf_paris_nettoyee,79367
1,dvf_paris_sans_dependances,26875
2,ev_actuels_nettoyes,2535
3,ev_supp_nettoyes,64
4,parcs_publics_nettoyee,511
